# Adaptive Retrieval Evaluation Audit

Run this notebook before introducing a higher-capacity retrieval or ranking algorithm. It evaluates Popularity, raw V4, and V4 with the serving popularity blend on one controlled point-in-time validation protocol. It reports Candidate Recall through Top-1000, exact-next coverage, bootstrap confidence intervals, and cold/warm plus history-length segments. No model is trained.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
WORK_ROOT = Path('/content/recsys_retrieval_audit')
SOURCE_ROOT = WORK_ROOT / 'runtime'
V4_DIR = WORK_ROOT / 'artifacts/v4_sequence_retrieval'
OUTPUT_DIR = WORK_ROOT / 'reports/retrieval_audit'
for path in [SOURCE_ROOT, V4_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('adaptive root:', ADAPTIVE_ROOT)
print('base Gold root:', BASE_GOLD_ROOT)

In [ ]:
required = {
    'pyarrow': 'pyarrow',
    'pandas': 'pandas',
    'torch': 'torch',
    'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import pandas as pd
import torch
assert torch.cuda.is_available(), 'Select an L4 or A100 GPU runtime before running the full audit.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    BASE_GOLD_ROOT / 'vocabularies/video_type',
    BASE_GOLD_ROOT / 'vocabularies/upload_type',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing inputs:\n' + '\n'.join(missing_paths)
assert (ADAPTIVE_ROOT / 'sequence_examples/train/_SUCCESS').exists()
assert (ADAPTIVE_ROOT / 'sequence_examples/validation/_SUCCESS').exists()
print('Drive preflight: PASS')

## Download the private runtime and frozen V4

Add a Colab Secret named `HF_TOKEN`, grant notebook access, and use a token that can read the private `khoa1505/recsys-artifacts` repository. No browser upload is required.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
V4_EXPECTED_SHA256 = '4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'
RUNTIME_EXPECTED_SHA256 = '1262393d30e21082849da6ae9ff74f1da4f4eac1d112860dac6a071909182327'
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a Colab Secret named HF_TOKEN and enable notebook access.') from exc
assert HF_TOKEN, 'HF_TOKEN is empty.'
runtime_zip = Path(hf_hub_download(repo_id=HF_REPO_ID, filename='runtime/adaptive_eval_runtime.zip', token=HF_TOKEN))
v4_cached = Path(hf_hub_download(repo_id=HF_REPO_ID, filename='v4/v4_inference_fp16.pt', token=HF_TOKEN))

def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
assert sha256(v4_cached) == V4_EXPECTED_SHA256, 'V4 checkpoint hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    archive.extractall(SOURCE_ROOT)
audit_script = SOURCE_ROOT / 'scripts/audit_adaptive_retrieval.py'
assert audit_script.exists(), f'Missing runtime script: {audit_script}'
v4_target = V4_DIR / 'best_model.pt'
if v4_target.exists() or v4_target.is_symlink():
    v4_target.unlink()
v4_target.symlink_to(v4_cached.resolve())
print('artifact integrity: PASS')
print('runtime:', runtime_zip)
print('V4:', v4_cached, f'{v4_cached.stat().st_size / 1024**3:.2f} GiB')

## Run the controlled audit

The full setting evaluates all latest validation users and the complete validation item catalog. Top-1000 is the retrieval gate; NDCG@10 is the primary ranking metric.

In [ ]:
command = [
    sys.executable, str(audit_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--split', 'validation',
    '--max-queries', '0',
    '--max-candidate-items', '0',
    '--top-ks', '10,50,100,500,1000',
    '--popularity-blend-grid', '0.0,0.02',
    '--bootstrap-samples', '1000',
    '--item-batch-size', '8192',
    '--query-batch-size', '32',
    '--device', 'cuda',
]
log_path = Path('/content/retrieval_audit.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w') as log_file:
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-120:])
    raise RuntimeError(f'Audit failed with exit code {return_code}. Last log lines:\n{tail}')

## Inspect the decision metrics

In [ ]:
report = json.loads((OUTPUT_DIR / 'retrieval_audit.json').read_text())
summary = pd.read_csv(OUTPUT_DIR / 'summary.csv')
segments = pd.read_csv(OUTPUT_DIR / 'segments.csv')
display(summary[[
    'variant', 'target_catalog_coverage', 'warm_target_rate',
    'hitrate@10', 'ndcg@10', 'recall@100', 'recall@500', 'recall@1000',
    'exact_hitrate@1000',
]])
display(segments[segments['variant'].eq('v4_popularity_0.02')][[
    'segment', 'queries', 'hitrate@10', 'ndcg@10', 'recall@1000', 'exact_hitrate@1000',
]])
for variant in summary['variant']:
    ci = report['bootstrap_ci95'][variant]['ndcg@10']
    print(variant, 'NDCG@10:', ci)

## Persist the lightweight report

Only reports and per-query metrics are saved. V4 weights and item embedding caches are excluded.

In [ ]:
archive_path = Path('/content/adaptive_retrieval_audit.zip')
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
    if log_path.exists():
        archive.write(log_path, 'logs/retrieval_audit.log')
persistent_dir = DRIVE_ROOT / 'artifacts/adaptive/retrieval_audit'
persistent_dir.mkdir(parents=True, exist_ok=True)
persistent_path = persistent_dir / archive_path.name
shutil.copy2(archive_path, persistent_path)
print('saved to Drive:', persistent_path, archive_path.stat().st_size, 'bytes')
files.download(str(archive_path))